这是比较老旧的版本，tensorflow 1，现在已经过时了。

In [1]:
import tensorflow as tf
import os
import pickle
import numpy as np

CIFAR_DIR = "./cifar-10-batches-py"
print(os.listdir(CIFAR_DIR))

/home/luke/.virtualenvs/tf1.13_py3/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/luke/.virtualenvs/tf1.13_py3/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/luke/.virtualenvs/tf1.13_py3/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/luke/.virtualenvs/tf1.13_py3/lib/python3

['data_batch_1', 'data_batch_5', 'data_batch_2', 'test_batch', 'batches.meta', 'readme.html', 'data_batch_4', 'data_batch_3']


In [2]:
train_filenames = [os.path.join(CIFAR_DIR, 'data_batch_%d' % i) for i in range(1, 6)]
test_filenames = [os.path.join(CIFAR_DIR, 'test_batch')]
train_filenames

['./cifar-10-batches-py/data_batch_1',
 './cifar-10-batches-py/data_batch_2',
 './cifar-10-batches-py/data_batch_3',
 './cifar-10-batches-py/data_batch_4',
 './cifar-10-batches-py/data_batch_5']

In [3]:
test_filenames 

['./cifar-10-batches-py/test_batch']

In [4]:
#加载数据集
def load_data(filename):
    """read data from data file."""
    with open(filename, 'rb') as f:
        data = pickle.load(f, encoding='bytes')
        return data[b'data'], data[b'labels']

# tensorflow.Dataset.
class CifarData:
    def __init__(self, filenames, need_shuffle):
        all_data = []
        all_labels = []
        for filename in filenames:#遍历每一个文件
            data, labels = load_data(filename)
            all_data.append(data)
            all_labels.append(labels)
        self._data = np.vstack(all_data)
        self._data = self._data / 127.5 - 1  #数据范围是变为-1到1之间
        self._labels = np.hstack(all_labels)
        print(self._data.shape)
        print(self._data[0])
        print(self._labels.shape)
        print(self._labels[0])
        
        self._num_examples = self._data.shape[0]  #存储样本数
        self._need_shuffle = need_shuffle
        self._indicator = 0  #索引从0开始
        if self._need_shuffle:
            self._shuffle_data()
            
    def _shuffle_data(self):
        # [0,1,2,3,4,5] -> [5,3,2,4,0,1]
        #         np.random.permutation(10)
        # array([1, 7, 4, 3, 0, 9, 2, 5, 8, 6]) #
        p = np.random.permutation(self._num_examples)
        self._data = self._data[p]
        self._labels = self._labels[p]
    
    def next_batch(self, batch_size):#手工分批操作
        """return batch_size examples as a batch."""
        end_indicator = self._indicator + batch_size
        if end_indicator > self._num_examples:#结束索引打印总样本数不再分批
            if self._need_shuffle:
                self._shuffle_data()
                self._indicator = 0
                end_indicator = batch_size
            else:
                raise Exception("have no more examples")
        if end_indicator > self._num_examples:
            raise Exception("batch size is larger than all examples")
        batch_data = self._data[self._indicator: end_indicator]  #拿某个批次的特征
        batch_labels = self._labels[self._indicator: end_indicator]
        self._indicator = end_indicator
        return batch_data, batch_labels

train_filenames = [os.path.join(CIFAR_DIR, 'data_batch_%d' % i) for i in range(1, 6)]
test_filenames = [os.path.join(CIFAR_DIR, 'test_batch')]

train_data = CifarData(train_filenames, True)  #True代表需要洗牌
test_data = CifarData(test_filenames, False)
print(type(train_data))
print(type(test_data))

(50000, 3072)
[-0.5372549  -0.6627451  -0.60784314 ...  0.09803922 -0.34117647
 -0.43529412]
(50000,)
6
(10000, 3072)
[ 0.23921569  0.24705882  0.29411765 ... -0.02745098  0.01176471
 -0.1372549 ]
(10000,)
3
<class '__main__.CifarData'>
<class '__main__.CifarData'>


In [ ]:
# 正常vgg是每一层通道数翻倍的
x = tf.placeholder(tf.float32, [None, 3072])
y = tf.placeholder(tf.int64, [None])
# [None], eg: [0,5,6,3]
x_image = tf.reshape(x, [-1, 3, 32, 32])  #为了跟存储的数据集进行匹配
# 32*32
x_image = tf.transpose(x_image, perm=[0, 2, 3, 1])
# print(x_image.shape)
# conv1: 神经元图， feature_map, 输出图像
conv1_1 = tf.layers.conv2d(x_image,
                           32, # output channel number
                           (3,3), # kernel size
                           padding = 'same',
                           activation = tf.nn.relu,
                           name = 'conv1_1')
conv1_2 = tf.layers.conv2d(conv1_1,
                           32, # output channel number
                           (3,3), # kernel size
                           padding = 'same',
                           activation = tf.nn.relu,
                           name = 'conv1_2')

# 16 * 16
pooling1 = tf.layers.max_pooling2d(conv1_2,
                                   (2, 2), # kernel size
                                   (2, 2), # stride
                                   name = 'pool1')


conv2_1 = tf.layers.conv2d(pooling1,
                           32, # output channel number
                           (3,3), # kernel size
                           padding = 'same',
                           activation = tf.nn.relu,
                           name = 'conv2_1')
conv2_2 = tf.layers.conv2d(conv2_1,
                           32, # output channel number
                           (3,3), # kernel size
                           padding = 'same',
                           activation = tf.nn.relu,
                           name = 'conv2_2')
# 8 * 8
pooling2 = tf.layers.max_pooling2d(conv2_2,
                                   (2, 2), # kernel size
                                   (2, 2), # stride
                                   name = 'pool2')

conv3_1 = tf.layers.conv2d(pooling2,
                           32, # output channel number
                           (3,3), # kernel size
                           padding = 'same',
                           activation = tf.nn.relu,
                           name = 'conv3_1')
conv3_2 = tf.layers.conv2d(conv3_1,
                           32, # output channel number
                           (3,3), # kernel size
                           padding = 'same',
                           activation = tf.nn.relu,
                           name = 'conv3_2')
# 4 * 4 * 32
pooling3 = tf.layers.max_pooling2d(conv3_2,
                                   (2, 2), # kernel size
                                   (2, 2), # stride
                                   name = 'pool3')
# [None, 4 * 4 * 32]
flatten = tf.layers.flatten(pooling3)
y_ = tf.layers.dense(flatten, 10)

#会先做softmax，然后再计算交叉熵损失
loss = tf.losses.sparse_softmax_cross_entropy(labels=y, logits=y_)
# y_ -> sofmax
# y -> one_hot
# loss = ylogy_

# indices
predict = tf.argmax(y_, 1)
# [1,0,1,1,1,0,0,0]
correct_prediction = tf.equal(predict, y)
accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float64))

# train_op里边的minimize做了compute_gradients()和 apply_gradients()  即计算梯度，更新梯度
with tf.name_scope('train_op'):
    train_op = tf.train.AdamOptimizer(1e-3).minimize(loss)

Instructions for updating:
Use keras.layers.conv2d instead.
Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Use keras.layers.max_pooling2d instead.
Instructions for updating:
Use keras.layers.flatten instead.
Instructions for updating:
Use keras.layers.dense instead.
Instructions for updating:
Use tf.cast instead.


In [ ]:
init = tf.global_variables_initializer()#tf1.0版本中要求做的，2.0不需要
batch_size = 20
train_steps = 10000
test_steps = 100

# train 10k: 73.4%  训练一万次后的效果
with tf.Session() as sess:
    sess.run(init)#2.0不需要
    for i in range(train_steps):
        batch_data, batch_labels = train_data.next_batch(batch_size)
        loss_val, acc_val, _ = sess.run(
            [loss, accuracy, train_op],
            feed_dict={
                x: batch_data,
                y: batch_labels})
        if (i+1) % 100 == 0:
            print('[Train] Step: %d, loss: %4.5f, acc: %4.5f' 
                  % (i+1, loss_val, acc_val))
        if (i+1) % 1000 == 0:# 每训练1000次，就在测试集上做一个验证
            test_data = CifarData(test_filenames, False)
            all_test_acc_val = []  #为了保存测试集上的准确率，最后求平均
            for j in range(test_steps):
                test_batch_data, test_batch_labels \
                    = test_data.next_batch(batch_size)
                test_acc_val = sess.run(  #测试集上的sess.run
                    [accuracy],
                    feed_dict = {
                        x: test_batch_data, 
                        y: test_batch_labels
                    })
                all_test_acc_val.append(test_acc_val)
            test_acc = np.mean(all_test_acc_val)
            print('[Test ] Step: %d, acc: %4.5f' % (i+1, test_acc))

[Train] Step: 100, loss: 1.71312, acc: 0.40000
[Train] Step: 200, loss: 1.60222, acc: 0.40000
[Train] Step: 300, loss: 1.73464, acc: 0.35000
[Train] Step: 400, loss: 1.39951, acc: 0.55000
[Train] Step: 500, loss: 1.61765, acc: 0.35000
[Train] Step: 600, loss: 1.74459, acc: 0.45000
[Train] Step: 700, loss: 1.55195, acc: 0.45000
[Train] Step: 800, loss: 1.62392, acc: 0.50000
[Train] Step: 900, loss: 1.83385, acc: 0.25000
[Train] Step: 1000, loss: 1.43746, acc: 0.40000
(10000, 3072)
[ 0.23921569  0.24705882  0.29411765 ... -0.02745098  0.01176471
 -0.1372549 ]
(10000,)
3
[Test ] Step: 1000, acc: 0.50050
[Train] Step: 1100, loss: 1.35813, acc: 0.55000
[Train] Step: 1200, loss: 1.39039, acc: 0.40000
[Train] Step: 1300, loss: 1.54810, acc: 0.40000
[Train] Step: 1400, loss: 1.28266, acc: 0.60000
[Train] Step: 1500, loss: 1.14216, acc: 0.60000
[Train] Step: 1600, loss: 1.24865, acc: 0.65000
[Train] Step: 1700, loss: 1.54039, acc: 0.55000
[Train] Step: 1800, loss: 0.85493, acc: 0.80000
[Train] 